In [1]:

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
from tqdm import tqdm
import numpy as np

In [2]:

ROI_SESSION1 = r"C:\Users\hiteshk\Desktop\Deep Learning Approaches for roi extraction and using same for palm print recognisation\Tonji\ROI\session1"
ROI_SESSION2 = r"C:\Users\hiteshk\Desktop\Deep Learning Approaches for roi extraction and using same for palm print recognisation\Tonji\ROI\session2"

CREASE_SESSION1 = r"C:\Users\hiteshk\Desktop\Deep Learning Approaches for roi extraction and using same for palm print recognisation\archive\blackandwhite_season1"
CREASE_SESSION2 = r"C:\Users\hiteshk\Desktop\Deep Learning Approaches for roi extraction and using same for palm print recognisation\archive\blackandwhite_season2"

GEN_SAVE_SESSION1 = r"C:\Users\hiteshk\Desktop\Deep Learning Approaches for roi extraction and using same for palm print recognisation\archive\genrois_session1"
GEN_SAVE_SESSION2 = r"C:\Users\hiteshk\Desktop\Deep Learning Approaches for roi extraction and using same for palm print recognisation\archive\genrois_session2"

os.makedirs(GEN_SAVE_SESSION1, exist_ok=True)
os.makedirs(GEN_SAVE_SESSION2, exist_ok=True)

print("✅ All folders ready!")

✅ All folders ready!


In [3]:

IMAGE_SIZE = 64

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

class PalmCreaseDataset(Dataset):
    def __init__(self, roi_dir, crease_dir):
        self.roi_dir = roi_dir
        self.crease_dir = crease_dir
        self.files = sorted([f for f in os.listdir(roi_dir) if f.lower().endswith(('.bmp', '.tiff'))])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        roi_name = self.files[idx]
        crease_name = roi_name.replace('.bmp', '.jpg').replace('.tiff', '.jpg')

        roi_path = os.path.join(self.roi_dir, roi_name)
        crease_path = os.path.join(self.crease_dir, crease_name)

        roi = Image.open(roi_path).convert("RGB")
        crease = Image.open(crease_path).convert("L")   # your black-white creases

        roi = transform(roi)
        crease = transform(crease) * 2.0 - 1.0

        x = torch.cat([roi, crease], dim=0)   # 4-channel input (exactly your PDF method)
        return x, roi_name

In [4]:
# CELL 4: Small UNet (fits in GTX 1650) + SSIM function
class ConditionalUNet(nn.Module):
    def __init__(self, in_channels=4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 4, 3, padding=1)
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

def calculate_ssim(img1, img2):
    img1 = (img1 + 1) / 2
    img2 = (img2 + 1) / 2
    mu1 = img1.mean(dim=[1,2], keepdim=True)
    mu2 = img2.mean(dim=[1,2], keepdim=True)
    sigma1 = ((img1 - mu1)**2).mean(dim=[1,2], keepdim=True)
    sigma2 = ((img2 - mu2)**2).mean(dim=[1,2], keepdim=True)
    sigma12 = ((img1 - mu1) * (img2 - mu2)).mean(dim=[1,2], keepdim=True)
    c1 = 0.01**2
    c2 = 0.03**2
    ssim_map = ((2*mu1*mu2 + c1) * (2*sigma12 + c2)) / ((mu1**2 + mu2**2 + c1) * (sigma1**2 + sigma2**2 + c2))
    return ssim_map.mean().item()

In [5]:
# CELL 5: Run everything for SESSION 1 (train + generate 3 ROIs per image + metrics)
print("=== STARTING SESSION 1 ===")
dataset1 = PalmCreaseDataset(ROI_SESSION1, CREASE_SESSION1)
loader1 = DataLoader(dataset1, batch_size=1, shuffle=True, num_workers=0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ConditionalUNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.cuda.amp.GradScaler()

# Training (30 epochs - you can change)
for epoch in range(30):
    model.train()
    total_loss = 0
    pbar = tqdm(loader1, desc=f"Session1 Epoch {epoch+1}/30")
    for batch, _ in pbar:
        batch = batch.to(device)
        noise = torch.randn_like(batch)
        noisy = batch + 0.1 * noise

        with torch.cuda.amp.autocast():
            pred = model(noisy)
            loss = F.mse_loss(pred, noise)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        total_loss += loss.item()
        pbar.set_postfix(loss=total_loss / (len(pbar)+1))

print("Training Session 1 finished!")

=== STARTING SESSION 1 ===


C:\Users\hiteshk\AppData\Local\Temp\ipykernel_19264\1094237920.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Session1 Epoch 1/30:   0%|          | 0/6000 [00:00<?, ?it/s]C:\Users\hiteshk\AppData\Local\Temp\ipykernel_19264\1094237920.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Session1 Epoch 30/30: 100%|██████████| 6000/6000 [03:13<00:00, 31.00it/s, loss=0.0462] 

Training Session 1 finished!


In [6]:

model.eval()
cos_sims1 = []
ssim_vals1 = []

with torch.no_grad():
    for orig_x, roi_name in tqdm(dataset1, desc="Generating 3 ROIs - Session 1"):
        orig_x = orig_x.unsqueeze(0).to(device)
        crease_channel = orig_x[:, 3:4, :, :]

        for gen_idx in range(3):
            noise = torch.randn_like(orig_x)
            noisy = orig_x + 0.1 * noise
            generated = model(noisy)

            gen_rgb = generated[:, :3, :, :].squeeze(0).cpu()
            orig_rgb = orig_x[:, :3, :, :].squeeze(0).cpu()

            # Save generated ROI
            save_name = roi_name.replace('.bmp','').replace('.tiff','') + f"_gen{gen_idx+1}.jpg"
            save_path = os.path.join(GEN_SAVE_SESSION1, save_name)
            pil_img = transforms.ToPILImage()((gen_rgb + 1) / 2)
            pil_img.save(save_path)

            # Metrics
            cos_sim = F.cosine_similarity(orig_rgb.flatten().unsqueeze(0), gen_rgb.flatten().unsqueeze(0), dim=1).item()
            cos_sims1.append(cos_sim)
            ssim_val = calculate_ssim(orig_rgb, gen_rgb)
            ssim_vals1.append(ssim_val)

avg_cos1 = np.mean(cos_sims1)
avg_ssim1 = np.mean(ssim_vals1)
print(f"\nSESSION 1 RESULTS:")
print(f"Average Cosine Similarity: {avg_cos1:.4f}")
print(f"Average SSIM: {avg_ssim1:.4f}")
print(f"Generated images saved in: {GEN_SAVE_SESSION1}")

Generating 3 ROIs - Session 1: 100%|██████████| 6000/6000 [01:40<00:00, 59.86it/s]


SESSION 1 RESULTS:
Average Cosine Similarity: 0.0219
Average SSIM: 0.0515
Generated images saved in: C:\Users\hiteshk\Desktop\Deep Learning Approaches for roi extraction and using same for palm print recognisation\archive\genrois_session1


In [7]:

print("=== STARTING SESSION 2 ===")
dataset2 = PalmCreaseDataset(ROI_SESSION2, CREASE_SESSION2)
loader2 = DataLoader(dataset2, batch_size=1, shuffle=True, num_workers=0)

model = ConditionalUNet().to(device)   # fresh model for Session 2
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.cuda.amp.GradScaler()

for epoch in range(20):
    model.train()
    total_loss = 0
    pbar = tqdm(loader2, desc=f"Session2 Epoch {epoch+1}/20")
    for batch, _ in pbar:
        batch = batch.to(device)
        noise = torch.randn_like(batch)
        noisy = batch + 0.1 * noise

        with torch.cuda.amp.autocast():
            pred = model(noisy)
            loss = F.mse_loss(pred, noise)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        total_loss += loss.item()
        pbar.set_postfix(loss=total_loss / (len(pbar)+1))

print("Training Session 2 finished!")

C:\Users\hiteshk\AppData\Local\Temp\ipykernel_19264\2172572224.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


=== STARTING SESSION 2 ===


Session2 Epoch 1/20:   0%|          | 0/6000 [00:00<?, ?it/s]C:\Users\hiteshk\AppData\Local\Temp\ipykernel_19264\2172572224.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Session2 Epoch 20/20: 100%|██████████| 6000/6000 [02:52<00:00, 34.88it/s, loss=0.0464] 

Training Session 2 finished!


In [8]:
# CELL 8: Generate 3 ROIs per original for SESSION 2 + save + metrics
model.eval()
cos_sims2 = []
ssim_vals2 = []

with torch.no_grad():
    for orig_x, roi_name in tqdm(dataset2, desc="Generating 3 ROIs - Session 2"):
        orig_x = orig_x.unsqueeze(0).to(device)
        for gen_idx in range(3):
            noise = torch.randn_like(orig_x)
            noisy = orig_x + 0.1 * noise
            generated = model(noisy)

            gen_rgb = generated[:, :3, :, :].squeeze(0).cpu()
            orig_rgb = orig_x[:, :3, :, :].squeeze(0).cpu()

            save_name = roi_name.replace('.bmp','').replace('.tiff','') + f"_gen{gen_idx+1}.jpg"
            save_path = os.path.join(GEN_SAVE_SESSION2, save_name)
            pil_img = transforms.ToPILImage()((gen_rgb + 1) / 2)
            pil_img.save(save_path)

            cos_sim = F.cosine_similarity(orig_rgb.flatten().unsqueeze(0), gen_rgb.flatten().unsqueeze(0), dim=1).item()
            cos_sims2.append(cos_sim)
            ssim_val = calculate_ssim(orig_rgb, gen_rgb)
            ssim_vals2.append(ssim_val)

avg_cos2 = np.mean(cos_sims2)
avg_ssim2 = np.mean(ssim_vals2)
print(f"\nSESSION 2 RESULTS:")
print(f"Average Cosine Similarity: {avg_cos2:.4f}")
print(f"Average SSIM: {avg_ssim2:.4f}")
print(f"Generated images saved in: {GEN_SAVE_SESSION2}")

print("\n🎉 ALL DONE! Check your genrois_session1 and genrois_session2 folders.")

Generating 3 ROIs - Session 2: 100%|██████████| 6000/6000 [01:58<00:00, 50.51it/s]


SESSION 2 RESULTS:
Average Cosine Similarity: 0.0249
Average SSIM: 0.0524
Generated images saved in: C:\Users\hiteshk\Desktop\Deep Learning Approaches for roi extraction and using same for palm print recognisation\archive\genrois_session2

🎉 ALL DONE! Check your genrois_session1 and genrois_session2 folders.
